# Caso de Estudio Big Data — Caracterización y distribución territorial del impacto del conflicto armado en Colombia

**Asignatura:** Big Data 2026-II · Ciencia de Datos · Universidad Externado de Colombia  
**Equipo (MABD01):** Julián Steven Duarte Gómez · Julián Camilo Jiménez Rodríguez · Juan Tomás Rincón Pinzón  
**Cliente institucional:** Unidad para la Atención y Reparación Integral a las Víctimas (**UARIV**)  
**Conjunto de datos:** *Cifras de Víctimas por Hechos — Municipal* (`datos.gov.co`, ID `9qih-4vkc`)  
**Dimensiones confirmadas:** 11.121.064 filas · 19 columnas · 3.330.779.150 bytes (3,33 GB / 3,10 GiB) · CSV UTF-8  
**Entorno:** Jupyter Notebook en el nodo maestro del clúster Hadoop (1 maestro + 2 trabajadores), Spark 3.5.x sobre YARN, datos en HDFS (`/caso/victimas_uariv.csv`, replicación 2).

---

## Objetivo del análisis

Implementar un *pipeline* de procesamiento distribuido por lotes (**batch**) para:

1. Caracterizar la **concentración espacial e histórica** de hechos victimizantes (por departamento y municipio) con corte a septiembre de 2026.
2. Evaluar la **brecha de atención humanitaria**: relación personas declaradas vs. personas ubicadas.
3. Enriquecer los datos mediante un **Broadcast Join** con la codificación municipal/regional del DANE para evaluar la afectación por región natural.

> **Este notebook es el código preliminar**: se ejecuta únicamente en el clúster (Spark sobre YARN y HDFS); no corre en una máquina personal sin el clúster.

---

## Mapa de capturas obligatorias (Anexo A1: C1 a C11)

Cada sección indica qué captura se obtiene en ella. Todas deben mostrar fecha/hora del sistema; las de Spark UI (C5, C7–C11) deben mostrar la URL con `application_<ts>_<id>`; las de notebook (C6, C11) la celda completa (código + salida).

| Captura | Entorno | Qué se captura |
|:--:|:--|:--|
| C1 | HDFS UI `:9870` → *Datanodes* | Dos DataNodes "In service" |
| C2 | HDFS UI `:9870` → *Browse* | Archivo en `/caso`, tamaño y **Replication = 2** |
| C3 | Terminal (`hadoop`) | `hdfs fsck /caso/victimas_uariv.csv -files -blocks -locations` |
| C4 | YARN UI `:8088` | `application_id` de referencia, usuario `hadoop` |
| C5 | Spark UI → *Environment* | *Spark Properties* (executor memory/cores, shuffle partitions) |
| C6 | Notebook | Salida de `.explain(True)` con `AdaptiveSparkPlan isFinalPlan=false` |
| C7 | Spark UI → *SQL/DataFrame* | Plan final con `isFinalPlan=true` |
| C8 | Spark UI → *Stages* | Stage relevante 1 (join o agregación departamental) |
| C9 | Spark UI → *Stages* | Stage relevante 2 (mayor duración) |
| C10 | Spark UI → *Storage* | DataFrame persistido (`Memory and Disk`, Fraction Cached 100%) |
| C11 | Spark UI / Notebook | SMJ vs. BHJ (eliminación del shuffle) |

---

## Imports y utilidades generales

Se importan **solo** las funciones necesarias (nunca `import *`): el `import *` sobrescribe funciones *built-in* de Python como `min()`, `max()` o `sum()`, que PySpark reemplaza por versiones pensadas para columnas, y produce errores como `min() takes 1 positional argument but 2 were given`.

También se define `fmt_co()`: formato numérico colombiano (punto para miles, coma para decimales) usado en las tablas y los gráficos. Es portable y no depende del *locale* del sistema operativo del nodo maestro.

In [ ]:
# =====================================================================
# Caso de Estudio Big Data — UARIV
# Cifras de Víctimas por Hechos Municipal (datos.gov.co ID: 9qih-4vkc)
# Se ejecuta en el nodo maestro del clúster Hadoop (1 maestro + 2 workers)
# con Spark 3.5.x sobre YARN y los datos en HDFS.
# =====================================================================

import time
import datetime

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, count, sum as spark_sum, round as spark_round,
    substring, to_date, year, broadcast, desc, lit
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType
)
from pyspark.storagelevel import StorageLevel


def fmt_co(valor, decimales=0, prefijo="$"):
    """Formatea un número con la convención colombiana:
    punto para miles y coma para decimales. Redondea a `decimales`.
    `prefijo` se antepone al número ("$" por defecto; "" para cantidades
    que no son dinero, como conteos).
    """
    signo = "-" if valor < 0 else ""
    texto = f"{abs(valor):,.{decimales}f}"   # formato anglosajón: 1,234,567.89
    texto = texto.replace(",", "_").replace(".", ",").replace("_", ".")
    return f"{signo}{prefijo}{texto}"


def encabezado(etiqueta):
    """Encabezado que acredita la ejecución propia: debe verse en cada captura."""
    print("=" * 72)
    print("{} | {:%Y-%m-%d %H:%M:%S}".format(etiqueta, datetime.datetime.now()))
    print("=" * 72)


print("Imports y utilidades cargados correctamente.")

---

## Sección 1 — Configuración de la SparkSession sobre YARN (Req. 2)

Se inicializa la sesión con parámetros explícitos adaptados al clúster físico (1 maestro + 2 trabajadores, 8 GB RAM / 4 cores por nodo):

- `spark.master = yarn`: YARN administra los recursos del clúster.
- `spark.executor.instances = 2`: un executor por cada nodo trabajador.
- `spark.executor.cores = 2`: evita contención de CPU en la máquina anfitriona.
- `spark.executor.memory = 2g`: deja margen al SO y a los servicios de Hadoop (NameNode, ResourceManager, DataNodes).
- `spark.sql.shuffle.partitions = 8`: reduce la sobrecarga de *scheduling* frente a las 200 particiones por defecto (8 particiones ÷ 4 cores = 2 por core).
- `spark.sql.adaptive.enabled = true`: habilita *Adaptive Query Execution* (AQE).

La salida debe mostrar `Master: yarn`. Si muestra `local[*]`, la sesión no corre en el clúster. **Captura C5**: la sección *Spark Properties* en Spark UI → *Environment* muestra estos parámetros modificados.

In [ ]:
encabezado("S1 · SparkSession sobre YARN")

spark = SparkSession.builder \
    .appName("CasoBD_UARIV_Victimas") \
    .master("yarn") \
    .config("spark.executor.instances", "2") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

print("SparkSession creada exitosamente.")
print("Versión de Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Application ID:", spark.sparkContext.applicationId)

---

## Sección 2 — Definición explícita del esquema (StructType)

**Justificación Big Data:** evitar `inferSchema=True`. Sobre 11,1 millones de filas y 3,33 GB, inferir el esquema exige un escaneo completo inicial sobre HDFS que **duplica el costo de I/O**. Con `StructType` se declaran los 19 campos una sola vez.

Tipos (según los metadatos Socrata del portal):

- `cod_pais`, `cod_estado_depto`, `cod_ciudad_muni`, `param_hecho` → `IntegerType` (códigos).
- `per_ocu`, `per_decla`, `per_ubic`, `per_sa`, `eventos` → `LongType` (conteos; se evita desbordamiento de `IntegerType`).
- `fecha_corte` → `StringType` (llega como `dd/MM/yyyy`; se convierte a fecha en la Sección 4).
- Las demás → `StringType`.

In [ ]:
encabezado("S2 · Esquema StructType")

schema = StructType([
    StructField("FECHA_CORTE",      StringType(),  True),
    StructField("NOM_RPT",           StringType(),  True),
    StructField("COD_PAIS",          IntegerType(), True),
    StructField("PAIS",              StringType(),  True),
    StructField("COD_ESTADO_DEPTO",  IntegerType(), True),
    StructField("ESTADO_DEPTO",      StringType(),  True),
    StructField("COD_CIUDAD_MUNI",   IntegerType(), True),
    StructField("CIUDAD_MUNICIPIO",  StringType(),  True),
    StructField("PARAM_HECHO",       IntegerType(), True),
    StructField("HECHO",             StringType(),  True),
    StructField("ETNIA",             StringType(),  True),
    StructField("SEXO",              StringType(),  True),
    StructField("DISCAPACIDAD",      StringType(),  True),
    StructField("CICLO_VITAL",       StringType(),  True),
    StructField("PER_OCU",           LongType(),    True),
    StructField("PER_DECLA",         LongType(),    True),
    StructField("PER_UBIC",          LongType(),    True),
    StructField("PER_SA",            LongType(),    True),
    StructField("EVENTOS",           LongType(),    True),
])

print("Esquema (StructType) definido con", len(schema), "campos.")

---

## Sección 3 — Ingesta y verificación de particiones

Lectura del CSV desde HDFS con el esquema explícito. La ruta usa `hdfs:///` (el `fs.defaultFS` del clúster apunta a HDFS).

Se inspecciona `df.rdd.getNumPartitions()` para cotejarlo con el número de bloques calculado en HDFS (capturas C2 y C3): Spark crea una partición por bloque HDFS (típicamente 25–27 bloques para 3,33 GB con bloques de 128 MB).

In [ ]:
encabezado("S3 · Ingesta desde HDFS")

RUTA_HDFS = "hdfs:///caso/victimas_uariv.csv"

t0 = time.time()
df = spark.read \
    .option("header", "true") \
    .option("sep", ",") \
    .schema(schema) \
    .csv(RUTA_HDFS)

print("Dataset leído (lazy) desde HDFS:", RUTA_HDFS)
print("Particiones del DataFrame:", df.rdd.getNumPartitions())
print("Número de columnas:", len(df.columns))
print("Tiempo de lectura (sin acción): {:.2f}s".format(time.time() - t0))
print()
df.printSchema()

---

## Sección 4 — Análisis exploratorio distribuido (Req. 3)

Tres verificaciones sobre el DataFrame completo:

1. **Conteo total** (`df.count()`): debe retornar **11.121.064**.
2. **Completitud**: nulos en `ESTADO_DEPTO`, `CIUDAD_MUNICIPIO`, `HECHO`, `ETNIA` y `SEXO`.
3. **Distribución por año de corte**: `FECHA_CORTE` viene como `dd/MM/yyyy`; se extrae el año y se agrega.

In [ ]:
encabezado("S4a · Conteo total de registros")

t0 = time.time()
total = df.count()
print("Total de registros:", fmt_co(total, 0, prefijo=""))
print("(Valor esperado: 11.121.064)")
print("Tiempo: {:.2f}s".format(time.time() - t0))

In [ ]:
encabezado("S4b · Completitud (nulos en columnas clave)")

columnas_clave = ["ESTADO_DEPTO", "CIUDAD_MUNICIPIO", "HECHO", "ETNIA", "SEXO"]

df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c + "_NULOS")
    for c in columnas_clave
]).show()

In [ ]:
encabezado("S4c · Distribución de registros por año de corte")

# Extracción robusta de año considerando las variaciones de formato en FECHA_CORTE:
# (ej. 'dd/MM/yyyy', 'yyyy/MM/dd', '30/09/24', 'dd/MM/yyyy 00:00')
df_anio = df.withColumn(
    "ANIO",
    when(col("FECHA_CORTE").contains("2024") | col("FECHA_CORTE").endswith("/24"), lit(2024))
    .when(col("FECHA_CORTE").contains("2025"), lit(2025))
    .when(col("FECHA_CORTE").contains("2026"), lit(2026))
    .otherwise(lit(None))
)

df_anio.groupBy("ANIO").agg(
    count("*").alias("REGISTROS"),
    spark_sum("EVENTOS").alias("EVENTOS")
).orderBy("ANIO").show()

In [ ]:
encabezado("S4d · Detalle de cortes de 2026 (Ajuste aprobado por la docente)")

# Número exacto de filas y suma de EVENTOS por cada corte de 2026
df_2026 = df.filter(col("FECHA_CORTE").contains("2026")) \
    .groupBy("FECHA_CORTE") \
    .agg(
        count("*").alias("FILAS"),
        spark_sum("EVENTOS").alias("SUMA_EVENTOS")
    ) \
    .orderBy("FECHA_CORTE")

df_2026.show(truncate=False)

---

## Sección 5 — Transformación compleja 1: agregación multidimensional (Req. 4)

Agrupación por `(ESTADO_DEPTO, CIUDAD_MUNICIPIO, HECHO)` (se conserva `COD_ESTADO_DEPTO` como clave del join posterior). Métricas calculadas:

- `TOTAL_EVENTOS`: suma de `EVENTOS` (impacto bruto).
- `TOTAL_DECLARANTES`: suma de `PER_DECLA` (personas que rindieron declaración).
- `TOTAL_UBICADOS`: suma de `PER_UBIC` (personas localizadas para atención).
- `TASA_EFECTIVIDAD_UBICACION = (TOTAL_UBICADOS / TOTAL_DECLARANTES) * 100`: índice derivado de la **brecha de atención humanitaria**.

El `groupBy` produce un *shuffle* (Exchange) de los 3,33 GB; aquí se concentra el mayor volumen de *Shuffle Write* (stage relevante para C8).

In [ ]:
encabezado("S5 · Agregación multidimensional")

df_agregado = df.groupBy(
    "COD_ESTADO_DEPTO", "ESTADO_DEPTO", "CIUDAD_MUNICIPIO", "HECHO"
).agg(
    spark_sum("EVENTOS").alias("TOTAL_EVENTOS"),
    spark_sum("PER_DECLA").alias("TOTAL_DECLARANTES"),
    spark_sum("PER_UBIC").alias("TOTAL_UBICADOS")
).withColumn(
    "TASA_EFECTIVIDAD_UBICACION",
    when(col("TOTAL_DECLARANTES").isNull() | (col("TOTAL_DECLARANTES") == 0), lit(None))
    .otherwise((col("TOTAL_UBICADOS") / col("TOTAL_DECLARANTES")) * 100)
)

print("Filas del resultado agregado:", fmt_co(df_agregado.count(), 0, prefijo=""))
print()
print("Top 10 (por TOTAL_EVENTOS):")
df_agregado.orderBy(desc("TOTAL_EVENTOS")).show(10, truncate=False)

---

## Sección 6 — Transformación compleja 2: Broadcast Join con tabla satélite (Req. 4)

Se crea una **tabla pequeña** de metadatos geográficos: el código DANE de cada departamento y su **región natural** (Andina, Caribe, Pacífica, Orinoquía, Amazonía), incluyendo el código 0 (Sin Definir).

**Técnica:** `F.broadcast(df_regiones)`. La tabla satélite mide menos de 1 MB; difundirla a todos los ejecutores elimina el *Shuffle Exchange* de la tabla masiva de víctimas. En el plan aparece `BroadcastExchange` en lugar de `Exchange` para la tabla grande.

In [ ]:
encabezado("S6a · Tabla satélite (DANE + región natural)")

# Codificación oficial DIVIPOLA / DANE de departamentos y su región natural.
datos_regiones = [
    (0,  "Sin Definir",             "Sin Definir"),
    (5,  "Antioquia",               "Andina"),
    (8,  "Atlántico",               "Caribe"),
    (11, "Bogotá D.C.",             "Andina"),
    (13, "Bolívar",                 "Caribe"),
    (15, "Boyacá",                  "Andina"),
    (17, "Caldas",                  "Andina"),
    (18, "Caquetá",                 "Amazonía"),
    (19, "Cauca",                   "Pacífica"),
    (20, "Cesar",                   "Caribe"),
    (23, "Córdoba",                 "Caribe"),
    (25, "Cundinamarca",            "Andina"),
    (27, "Chocó",                   "Pacífica"),
    (41, "Huila",                   "Andina"),
    (44, "La Guajira",              "Caribe"),
    (47, "Magdalena",               "Caribe"),
    (50, "Meta",                    "Orinoquía"),
    (52, "Nariño",                  "Pacífica"),
    (54, "Norte de Santander",      "Andina"),
    (63, "Quindío",                 "Andina"),
    (66, "Risaralda",               "Andina"),
    (68, "Santander",               "Andina"),
    (70, "Sucre",                   "Caribe"),
    (73, "Tolima",                  "Andina"),
    (76, "Valle del Cauca",         "Pacífica"),
    (81, "Arauca",                  "Orinoquía"),
    (85, "Casanare",                "Orinoquía"),
    (86, "Putumayo",                "Amazonía"),
    (88, "San Andrés y Providencia", "Caribe"),
    (91, "Amazonas",                "Amazonía"),
    (94, "Guainía",                 "Amazonía"),
    (95, "Guaviare",                "Amazonía"),
    (97, "Vaupés",                  "Amazonía"),
    (99, "Vichada",                 "Orinoquía"),
]

df_regiones = spark.createDataFrame(
    datos_regiones,
    schema=["COD_ESTADO_DEPTO", "DEPARTAMENTO", "REGION_NATURAL"]
)

print("Tabla satélite creada con", df_regiones.count(), "departamentos.")
df_regiones.show(truncate=False)

In [ ]:
encabezado("S6b · Join enriquecido (Broadcast Hash Join)")

df_enriquecido = df_agregado.join(
    broadcast(df_regiones),
    on="COD_ESTADO_DEPTO",
    how="left"
)

print("Join (broadcast) completado.")
print("Filas del resultado enriquecido:",
      fmt_co(df_enriquecido.count(), 0, prefijo=""))

---

## Sección 7 — Bonificación (+0,3): Sort Merge Join vs. Broadcast Hash Join (Req. VIII)

Se comparan empíricamente las dos estrategias de *join* físico sobre la misma operación (tabla grande agregada ⋈ tabla satélite pequeña):

- **Escenario A (Sort Merge Join forzado):** `spark.sql.autoBroadcastJoinThreshold = -1` desactiva el *broadcast*; Spark redistribuye **ambas** tablas (shuffle + ordenamiento). El plan muestra `SortMergeJoin` y dos `Exchange`.
- **Escenario B (Broadcast Hash Join):** se restaura el umbral (10 MiB) y se usa el *hint* `broadcast()`. El plan muestra `BroadcastHashJoin` sin `Exchange` de la tabla grande.

**Evidencia C11:** en Spark UI → *Stages* se comparan *Shuffle Write/Read*, demostrando la reducción a cero del shuffle en el escenario B.

In [ ]:
encabezado("S7a · Escenario A — Sort Merge Join (broadcast desactivado)")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
print("Broadcast deshabilitado (threshold = -1) → se fuerza Sort Merge Join\n")

t0 = time.time()
df_smj = df_agregado.join(df_regiones, on="COD_ESTADO_DEPTO", how="left")
n_smj = df_smj.count()
tiempo_smj = time.time() - t0

print("Sort Merge Join completado en {:.2f}s ({} filas)".format(tiempo_smj, n_smj))
print()
print("Plan físico (buscar SortMergeJoin + Exchange):")
df_smj.explain()

In [ ]:
encabezado("S7b · Escenario B — Broadcast Hash Join (broadcast restaurado)")

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")  # 10 MiB
print("Broadcast habilitado (threshold = 10 MiB) → Broadcast Hash Join\n")

t0 = time.time()
df_bhj = df_agregado.join(
    broadcast(df_regiones), on="COD_ESTADO_DEPTO", how="left"
)
n_bhj = df_bhj.count()
tiempo_bhj = time.time() - t0

print("Broadcast Hash Join completado en {:.2f}s ({} filas)".format(tiempo_bhj, n_bhj))
print()
print("Plan físico (buscar BroadcastHashJoin, sin Exchange de la tabla grande):")
df_bhj.explain()

In [ ]:
encabezado("S7c · Comparación de rendimiento")

print("COMPARACIÓN SMJ vs BHJ")
print("=" * 60)
print("Sort Merge Join:      {:.2f}s".format(tiempo_smj))
print("Broadcast Hash Join:  {:.2f}s".format(tiempo_bhj))
if tiempo_bhj > 0:
    print("Speedup: {:.2f}x más rápido con broadcast".format(tiempo_smj / tiempo_bhj))
print()
print("Explicación:")
print("  • SMJ mueve AMBAS tablas por la red (shuffle + ordenamiento).")
print("  • BHJ copia solo la tabla pequeña una vez a cada executor (sin shuffle).")

---

## Sección 8 — Análisis del plan de ejecución y AQE (Req. 5)

Sobre la **consulta principal** (join enriquecido + agregación por región) se ejecuta `.explain(True)`, que muestra las 4 fases de Catalyst: *Parsed → Analyzed → Optimized → Physical*.

- **Captura C6:** la salida textual muestra `AdaptiveSparkPlan isFinalPlan=false` (plan **inicial**, antes de la retroalimentación de métricas de ejecución).
- Al ejecutar la acción siguiente, AQE genera el plan **final** (`isFinalPlan=true`), visible en **C7** (Spark UI → *SQL/DataFrame*).

Operadores a identificar: `FileScan csv`, `HashAggregate`, `Exchange` (shuffle), `BroadcastExchange` (broadcast) y `AdaptiveSparkPlan`.

In [ ]:
encabezado("S8a · Plan extendido de la consulta principal (isFinalPlan=false)")

consulta_principal = df_agregado.join(
    broadcast(df_regiones), on="COD_ESTADO_DEPTO", how="left"
).groupBy("REGION_NATURAL").agg(
    spark_sum("TOTAL_EVENTOS").alias("EVENTOS_REGION"),
    spark_sum("TOTAL_DECLARANTES").alias("DECLARANTES_REGION"),
    spark_sum("TOTAL_UBICADOS").alias("UBICADOS_REGION")
)

print("Plan de ejecución extendido (plan inicial AQE):")
print("=" * 72)
consulta_principal.explain(True)
print("=" * 72)

In [ ]:
encabezado("S8b · Acción que dispara el plan final (isFinalPlan=true)")

t0 = time.time()
consulta_principal.cache()   # se reutilizará en la Sección 9/10
consulta_principal.count()
print("Acción ejecutada en {:.2f}s".format(time.time() - t0))
print("Ver el plan FINAL (isFinalPlan=true) en Spark UI → SQL/DataFrame (captura C7).")

---

## Sección 9 — Persistencia y gestión de memoria (Req. 8)

Se aplica `.persist(StorageLevel.MEMORY_AND_DISK)` al DataFrame consolidado porque se ejecutarán **dos acciones consecutivas** sobre él:

1. **Acción 1:** agrupación por región natural (reporte macro).
2. **Acción 2:** materialización del *top* municipal para la visualización ejecutiva (Sección 10).

**Captura C10:** en Spark UI → *Storage* debe verse `Storage Level: Memory and Disk`, `Fraction Cached: 100%`, `Size in Memory` y `Size on Disk`. Al final se libera con `.unpersist()`.

In [ ]:
encabezado("S9 · Persistencia del DataFrame consolidado")

df_consolidado = df_enriquecido.persist(StorageLevel.MEMORY_AND_DISK)

# Acción 1: materializar el cache
n = df_consolidado.count()
print("Materializado (Acción 1):", fmt_co(n, 0, prefijo=""), "filas")
print("Storage Level:", df_consolidado.storageLevel)
print("Particiones:", df_consolidado.rdd.getNumPartitions())
print()
print("Ver Storage tab en Spark UI (captura C10).")

In [ ]:
encabezado("S9b · Acción 2 — reporte por Región Natural")

df_region = df_consolidado.groupBy("REGION_NATURAL").agg(
    spark_sum("TOTAL_EVENTOS").alias("EVENTOS_REGION"),
    spark_sum("TOTAL_DECLARANTES").alias("DECLARANTES_REGION"),
    spark_sum("TOTAL_UBICADOS").alias("UBICADOS_REGION")
).orderBy(desc("EVENTOS_REGION"))

df_region.show(truncate=False)

---

## Sección 10 — Materialización (.toPandas) y visualización (Req. 7)

Se aplica el patrón Big Data: **Procesamiento distribuido → Agregación → Reducción a ≤ 35 filas → `.toPandas()`**.

> ⚠️ Nunca se llama `.toPandas()` sobre el DataFrame crudo de millones de filas: colapsaría la memoria del driver. Siempre se agrega primero y se verifica el número de filas.

Los gráficos usan `matplotlib` con la paleta **Okabe-Ito** (accesible para daltónicos) y formato colombiano (`fmt_co()`) en etiquetas y ejes.

In [ ]:
encabezado("S10a · Reducción a ≤ 35 filas: top 10 municipios por EVENTOS")

df_top10 = df_consolidado.groupBy("ESTADO_DEPTO", "CIUDAD_MUNICIPIO").agg(
    spark_sum("TOTAL_EVENTOS").alias("EVENTOS")
).orderBy(desc("EVENTOS")).limit(10)

print("Filas del resultado agregado (antes de .toPandas()):", df_top10.count())
df_top10.show(truncate=False)

In [ ]:
encabezado("S10b · Conversión a pandas (solo el resultado agregado)")

pdf_top10 = df_top10.toPandas()
pdf_region = df_region.toPandas()

print("top10 →", pdf_top10.shape, "· regiones →", pdf_region.shape)

In [ ]:
encabezado("S10c · Gráfico 1 — Top 10 municipios por hechos victimizantes")

import matplotlib.pyplot as plt

azul_oi = "#0072B2"

fig, ax = plt.subplots(figsize=(11, 6))
pdf_orden = pdf_top10.iloc[::-1]  # el mayor arriba

barras = ax.barh(
    pdf_orden["CIUDAD_MUNICIPIO"], pdf_orden["EVENTOS"],
    color=azul_oi
)
ax.bar_label(barras,
             labels=[fmt_co(v, 0, prefijo="") for v in pdf_orden["EVENTOS"]],
             fontsize=8, padding=2)

ax.set_title("Top 10 municipios por hechos victimizantes (UARIV, corte sep-2026)")
ax.set_xlabel("Número de hechos (EVENTOS)")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: fmt_co(v, 0, prefijo="")))
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
encabezado("S10d · Gráfico 2 — Hechos por Región Natural")

naranja_oi = "#E69F00"

fig, ax = plt.subplots(figsize=(9, 5))
pdf_region_orden = pdf_region.sort_values("EVENTOS_REGION", ascending=True)

barras = ax.barh(
    pdf_region_orden["REGION_NATURAL"], pdf_region_orden["EVENTOS_REGION"],
    color=naranja_oi
)
ax.bar_label(barras,
             labels=[fmt_co(v, 0, prefijo="") for v in pdf_region_orden["EVENTOS_REGION"]],
             fontsize=9, padding=3)

ax.set_title("Hechos victimizantes por Región Natural (UARIV, corte sep-2026)")
ax.set_xlabel("Número de hechos (EVENTOS)")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: fmt_co(v, 0, prefijo="")))
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
encabezado("S10e · Liberación de memoria y cierre")

df_consolidado.unpersist()
print("Cache liberado (unpersist).")
print()
print("Análisis principal finalizado. SparkSession queda abierta")
print("para consultar la Spark UI y capturar C8, C9, C10 y C11.")
# Nota: NO ejecutar spark.stop() hasta terminar todas las capturas de Spark UI.

---

## Siguientes pasos en el clúster

1. **Sesión 1 (lunes 9/nov):** verificar `df -h`, cargar a HDFS con replicación 2, generar el `application_id` de referencia en YARN. Capturas C1–C5.
2. **Sesión 2 (miércoles 11/nov):** ejecutar el análisis principal (agregaciones + join), capturas C6–C9, persistencia (C10) y prueba SMJ vs BHJ (C11).
3. **Sesión 3 (viernes 13/nov):** verificar estado del clúster y capturas faltantes o de incidentes.

Registrar los valores observados en el **Anexo A2** (almacenamiento HDFS, SparkSession, exploración, plan de ejecución, métricas de stages, resultados/cache).